In [3]:
import pandas as pd
import numpy as np
import sklearn as sk
import os

In [4]:
os.getcwd()

'c:\\Users\\Hall\\AppData\\Local\\Programs\\Microsoft VS Code'

In [ ]:
#os.chdir('C:\\Users\\Hall\\Downloads')
# Just make sure you are in the directory with the ipc fils

In [6]:
#Pull in ipc_train, ipc_test, and ipc_val csvs and combine
ipc_train = pd.read_csv('ipc_train.csv')
ipc_test = pd.read_csv('ipc_test.csv')
ipc_val = pd.read_csv('ipc_val.csv')


In [7]:
# Combine ipc_train and ipc_val
ipc_train_val = pd.concat([ipc_train, ipc_val], axis=0)
#Comebine ipc_train_val and ipc_test
ipc_train_val_test = pd.concat([ipc_train_val, ipc_test], axis=0)

In [8]:
ipc_train_val_test.head()

,number,link,text,user,likes,quotes,retweets,comments,label,text_clean,text_normalized,text_length,label_id
0,651,https://twitter.com/KabalexChild/status/175195...,The reality is that I'm in Fury's head: Usyk m...,News Ukraine,0,0,0,0,2,The reality is that I'm in Fury's head: Usyk m...,The reality is that I'm in Fury's head: Usyk m...,114,2
1,557,https://twitter.com/d1rogue/status/17850818427...,@WFP @fiannafailparty @FineGael @pb4p @sinnfei...,audge,0,0,0,0,1,@WFP @fiannafailparty @FineGael @pb4p @sinnfei...,@USER @USER @USER @USER @USER Can you help.......,59,1
2,1099,https://twitter.com/OneMugManTwo/status/177457...,I would be in favor of a ceasefire as long as ...,William T. Garrett,0,0,0,0,3,I would be in favor of a ceasefire as long as ...,I would be in favor of a ceasefire as long as ...,74,3
3,13,https://twitter.com/iioannides/status/17521185...,There is also the @EU_Commission statement on ...,Isabelle Ioannides,0,0,0,5,1,There is also the @EU_Commission statement on ...,There is also the @USER statement on #UNRWA & ...,80,1
4,695,https://twitter.com/onekenneth/status/17519329...,Mecha whale flying in space - via MJ #scifi...,KEn ThE SEarchEr,2,0,0,0,2,Mecha whale flying in space - via MJ #scifi...,Mecha whale flying in space - via MJ #scifi #w...,240,2


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

#For reproducability if we need it
RANDOM_STATE = 42

# 1) Split dataset first (re-stratify by label)
# WE may be able to use the original splits, but I wasn't sure, I am following a tutorial
train_df, temp_df = train_test_split(ipc_train_val_test, test_size=0.2, stratify=ipc_train_val_test['label'], random_state=RANDOM_STATE)
val_df, test_df  = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=RANDOM_STATE)

# 2) Choose target count per class - bring all classes up to the largest class count in training set
class_counts = train_df['label'].value_counts()
print("Train class counts before oversample:\n", class_counts)

target_count = class_counts.max()   # or choose another target like int(class_counts.mean()*k)

# 3) Oversample each minority class with replacement
def oversample_dataframe(df, label_col='label', text_col='text', target_count=None, random_state=RANDOM_STATE):
    if target_count is None:
        target_count = df[label_col].value_counts().max()

    parts = []
    for cls, cnt in df[label_col].value_counts().items():
        cls_rows = df[df[label_col] == cls]
        if cnt < target_count:
            # sample with replacement
            n_needed = target_count - cnt
            sampled = cls_rows.sample(n=n_needed, replace=True, random_state=random_state)
            cls_rows = pd.concat([cls_rows, sampled], axis=0)
        # else leave as-is
        parts.append(cls_rows)
    new_df = pd.concat(parts).sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    return new_df

train_df_bal = oversample_dataframe(train_df, target_count=target_count)
print("Train class counts after oversample:\n", train_df_bal['label'].value_counts())



Train class counts before oversample:
 label
1    2262
2    1218
0    1140
4     764
5     480
3      31
Name: count, dtype: int64
Train class counts after oversample:
 label
1    2262
4    2262
5    2262
2    2262
0    2262
3    2262
Name: count, dtype: int64


In [10]:
train_df_bal.to_csv("train_ipc_oversampled.csv", index=False)
val_df.to_csv("val_ipc_balanced.csv", index=False)
test_df.to_csv("test_ipc_balanced.csv", index=False)